## 1. Prepare all Nigeria H3 Cells  at Resolution 8

### 00. Step Up

In [ ]:
%load_ext autoreload
%autoreload 2 

import sys
from pathlib import Path 
import logging 
from datetime import datetime
import pickle
import h3
from codebase.utils.utils import setup_logging 
import sys 
from config.settings import STORAGE_CONFIG, ADMIN_DATA_SOURCES, INPUT_BASE_DATA_SOURCES, RAW_DATA_DIR,PROCESSED_DATA_DIR, EXPORTS_DIR, OUTPUT_DIR
import pandas as pd
H3_DUCKDB_PATH = STORAGE_CONFIG['h3_duckdb_path'] 


logger = setup_logging(log_dir='log-main-sp-clustering-and-routing', 
                       projname='log-main-spcr')

### 01. Load and Preprocess Boundary Data

In [ ]:
%load_ext autoreload
%autoreload 2 


import sys
from pathlib import Path
project_root = Path('').parent
sys.path.append(str(project_root))


from pathlib import Path 
import logging 
from datetime import datetime
import pickle
from src.h3_spatial_system.data.downloader import DataDownloader, validate_and_summarize_data
from src.h3_spatial_system.h3_system.generator import H3AddressGenerator
from src.h3_spatial_system.storage.duckdb_storage import DuckDBStorage
from config.settings import * 


validate_downloaded_data with the function validate_and_summarize_data save the standardized geojson file to disk

In [ ]:
# Step 1: Download administrative boundary data
'''
logger.info("📥 Step 1: Downloading administrative boundary data...")
downloader = DataDownloader()
downloaded_files = downloader.download_admin_boundaries()
if downloaded_files:
    validation_results, summaries = validate_and_summarize_data(downloaded_files)
    
if not downloaded_files:
    logger.error("❌ Failed to download administrative boundary data")
    # return 1

logger.info(f"✅ Downloaded {len(downloaded_files)} boundary files")

'''

# load geojson files
dict_path_geojson ={
    'states': RAW_DATA_DIR / 'grid3-nga-operational-state-boundaries_standardized.geojson',
    'lgas': RAW_DATA_DIR / 'grid3-nga-operational-lga-boundaries_standardized.geojson',
    'wards': RAW_DATA_DIR / 'grid3-nga-operational-wards-v1-0_standardized.geojson'
}

logger.info("🚀 Starting H3-based address system generation for Nigeria")

# Step 2: Initialize H3 address generator
logger.info("🔧 Step 2: Initializing H3 address generator...")
generator = H3AddressGenerator(resolution=H3_RESOLUTION)

In [ ]:
# Step 3: Load administrative boundaries
logger.info("🗺️ Step 3: Loading administrative boundaries...")
states_path = str(dict_path_geojson['states'])
lgas_path = str(dict_path_geojson['lgas'])
wards_path = str(dict_path_geojson['wards'])

generator.load_admin_boundaries(states_path, lgas_path, wards_path)
logger.info("✅ Administrative boundaries loaded")

In [ ]:
# ADMIN_DATA_SOURCES
str({key:value['standardize_file_path'] for key, value in ADMIN_DATA_SOURCES.items()}['states'])

### 02. Generate H3 Cells

In [ ]:
# Step 4: Generate H3 cells
logger.info("🔷 Step 4: Generating H3 cells...")
h3_cells = generator.generate_h3_cells() 
with open(PROCESSED_DATA_DIR'h3_cells_res8.pickle', 'wb') as f:
    pickle.dump(h3_cells, f)

### 03. Add Meta data to H3 Cells

In [ ]:
%load_ext autoreload
%autoreload 2 


import sys
from pathlib import Path
import logging 

import h3
import pickle
from src.h3_spatial_system.h3_system.utils import h3_to_objects_parallel_safe, h3_to_objects_parallel_generator, H3SQLiteManager

In [ ]:
# Load Data
with open('./data/processed/h3_cells_res8.pickle', 'rb') as f:
    h3_cells = pickle.load(f)
logger.info(f"✅ Generated {len(h3_cells):,} H3 cells")

In [ ]:
# import time 
# start_time = time.time()   
# h3_cells_processed =  h3_to_objects_parallel_safe(h3_cells[0:1000]) 
# parallel_time = time.time() - start_time 
# print(f"Total time taken {len(h3_cells):,} cells took: {parallel_time:.2f} seconds") 

In [ ]:
# # For very large datasets, use batched processing:
# h3_data = {}
# for h3_index, result in h3_to_objects_parallel_generator(h3_cells):
#     h3_data[h3_index] = result ## 9m 17s

# with open('./data/processed/h3_cells_processed_dict.pickle', 'wb') as f:
#     pickle.dump(h3_data, f)



# METHOD 2
# Stream directly to JSONL - most efficient
# Process and save your H3 data
# import time 
# start_time = time.time()   
# with H3SQLiteManager("./data/processed/h3_data.db") as db:
#     for h3_index, result in h3_to_objects_parallel_generator(h3_cells):
#         db.add_result(h3_index, result)

# parallel_time = time.time() - start_time 
# print(f"Total time taken {len(h3_cells):,} cells took: {parallel_time:.2f} seconds") 
# print("Done! Database saved with compression.")

In [ ]:
from src.h3_spatial_system.h3_system.FastH3DuckDBManager import FastH3DuckDBManager
from src.h3_spatial_system.h3_system.save_h3_data_utils import verify_db_table_and_h3_cells_data    
from config.settings import STORAGE_CONFIG

H3_DUCKDB_PATH = STORAGE_CONFIG['h3_duckdb_path']


# METHOD 3
# Stream directly to DUCKDB - most efficient
with FastH3DuckDBManager(resolution=8, db_path=H3_DUCKDB_PATH, batch_size=100000) as db:
    count = 0
    for h3_index, result in h3_to_objects_parallel_generator(h3_cells):
        db.add_result(h3_index, result)
        count += 1
        
        # Check database periodically to verify commits
        if count % 100000 == 0:
            stats = db.get_stats()
            print(f"Processed: {count:,}, In DB: {stats['total_records']:,}")

# # After the context manager exits, check final state
# verify_db_table_and_h3_cells_data(db_path)

In [ ]:
# Check if your current database file exists and has data
from src.h3_spatial_system.h3_system.save_h3_data_utils import verify_db_table_and_h3_cells_data
H3_DUCKDB_PATH = STORAGE_CONFIG['h3_duckdb_path'] 
verify_db_table_and_h3_cells_data(H3_DUCKDB_PATH)

In [ ]:
# print(len(h3_data))
# print(len(h3_data.keys()))

### 04. Generate Address and Address ID for the Cells & ADD TO DB

In [ ]:
%load_ext autoreload
%autoreload 2 

from src.h3_spatial_system.h3_system.FastH3DuckDBManager import FastH3DuckDBManager
from src.h3_spatial_system.h3_system.save_h3_data_utils import verify_db_table_and_h3_cells_data   
from config.settings import STORAGE_CONFIG 

H3_DUCKDB_PATH = STORAGE_CONFIG['h3_duckdb_path']

In [ ]:
# # Step 5: Generate addresses - to-do optu
# start_time = time.time() 
# logger.info("🏠 Step 5: Generating addresses...") 
# addresses = generator.generate_addresses(h3_cells) 
# logger.info(f"✅ Generated {len(addresses)} address records") 
# parallel_time = time.time() - start_time 
# print(f"Total time taken {len(h3_cells):,} cells took: {parallel_time:.2f} seconds") 


# To be Implemented
# This can handle both new and existing H3 indices
# with FastH3DuckDBManager(resolution=8, db_path=db_path) as db:
    # address_data = generator.generate_addresses(h3_cells) 
    # db.upsert_address_data(address_data)

In [ ]:
# Verifing DB
verify_db_table_and_h3_cells_data(H3_DUCKDB_PATH)

In [ ]:
# Added New addrss columns || Alternatively I can use raw sql
with FastH3DuckDBManager(resolution=8, db_path=H3_DUCKDB_PATH) as db:
    db._add_address_columns()

In [ ]:
import duckdb 
conn = duckdb.connect(H3_DUCKDB_PATH)

In [ ]:
conn.execute("SELECT count(distinct h3_index) FROM h3_cells where state_code is NULL").fetchdf()  #803,827

In [ ]:
import pandas as pd
PATH_H3_ADDRESS_DF = EXPORTS_DIR/ 'df_filterd_h3_res8_nigeria_addresses.parquet' 

df_address_filtered = duckdb.sql(f"SELECT * EXCLUDE __index_level_0__ FROM '{PATH_H3_ADDRESS_DF}'").fetchdf() 
df_address_filtered.rename(columns={'h3_id':'h3_index'}, inplace=True)
df_address_filtered['resolution'] = 8 
df_address_filtered.head(1)
print(f'Total Records: {len(df_address_filtered):,}')

In [ ]:
# Set your chunk size
batch_size = 100000
total_processed = 0

# Loop through the DataFrame in chunks
for i in range(0, len(df_address_filtered), batch_size):
    batch_data = df_address_filtered.iloc[i:i + batch_size]
    # print(f"Chunk {i // batch_size + 1}:\n", batch_data.head(), "\n")

    try: 
        conn.register('address_upsert_df', batch_data) 
        # Use INSERT OR REPLACE for UPSERT
        conn.execute("""
            INSERT OR REPLACE INTO h3_cells (
                h3_index, resolution, 
                h3_derived_id, grid_position_id, primary_address_id,
                country_code, country_name, state_code, state_name,
                lga_code, lga_name, ward_code, ward_name,
                confidence_level, coverage_percentage, area_km2
            )
            SELECT 
                h3_index, resolution,
                h3_derived_id, grid_position_id, primary_address_id,
                country_code, country_name, state_code, state_name,
                lga_code, lga_name, ward_code, ward_name,
                confidence_level, coverage_percentage, area_km2
            FROM address_upsert_df
        """)
        
        conn.unregister('address_upsert_df')
        total_processed += len(batch_data)
        
        print(f"✅ Upserted batch {i//batch_size + 1}: {len(batch_data):,} records (total: {total_processed:,})")
        
    except Exception as e:
        print(f"❌ Error upserting : {e}")
        # print(f"❌ Error upserting batch {i//batch_size + 1}: {e}")
        # continue
    

# print(f"🎉 Address data upsert completed: {total_processed:,} records processed")

In [ ]:
conn.close()

### 05. Evaluation with Plotting

In [ ]:
# pip install shiny shinywidgets hvplot geoviews geopandas hvplot holoviews bokeh shapely

In [ ]:
%load_ext autoreload
%autoreload 2 


import duckdb 
import geopandas as gpd
import pandas as pd
from pathlib import Path
from config.settings import STORAGE_CONFIG 
from src.h3_system.plot_utils import plot_h3_from_db #, plot_h3_from_db_fast

H3_DUCKDB_PATH = STORAGE_CONFIG['h3_duckdb_path']
test_map_path = Path('./output/map/test/')

In [ ]:
conn = duckdb.connect(H3_DUCKDB_PATH)
# Schema
# conn.execute('DESCRIBE h3_cells;').fetchdf()[['column_name', 'column_type', 'null', 'key']]

In [ ]:
df_sample_h3_with_metadata = gpd.GeoDataFrame(conn.execute('SELECT * FROM h3_cells WHERE confidence_level IS NOT NULL LIMIT 100').fetch_df())
# print(df_samole_h3_with_metadata.columns)
df_sample_h3_with_metadata.sample(2)

# ['h3_index', 'resolution', 'centroid_lat', 'centroid_lng', 'polygon_wkt',
# 'boundary_json', 'latlng_json', 'polygon_area', 'num_vertices', 'error',
# 'created_at', 'h3_derived_id', 'grid_position_id', 'primary_address_id',
# 'country_code', 'country_name', 'state_code', 'state_name', 'lga_code',
# 'lga_name', 'ward_code', 'ward_name', 'confidence_level',
# 'coverage_percentage', 'area_km2']

In [ ]:
print(conn.execute("SELECT DISTINCT confidence_level FROM h3_cells WHERE state_name = 'Lagos' AND confidence_level IS NOT NULL").fetch_df())
# boundary_case, confident


# Example H3 cell IDs
h3_cells_to_plot =conn.execute("""SELECT DISTINCT h3_index FROM h3_cells 
                               WHERE state_name = 'Lagos' 
                               AND confidence_level IS NOT NULL
                               AND lga_name = 'Apapa'
                              --- AND ward_name = 'Abraham Adesanya'
                               """).fetch_df().h3_index.to_list() 

##### 1. Ploting with folium

In [ ]:
plot_folium = plot_h3_from_db(h3_cells_to_plot, H3_DUCKDB_PATH, show_markers=False, 
                popup_fields=['h3_derived_id', 'grid_position_id','state_name','lga_name', 'ward_name', 'confidence_level', 'coverage_percentage'
                              #,'centroid_lat', 'centroid_lng'
                              ],
                colors =  ['#3388ff']*len(h3_cells_to_plot),
                # colors = None,
                polygon_weight = 1,
                polygon_opacity = 0.05
                )

plot_folium

In [ ]:
plot_folium.save(test_map_path / 'test_save_as_folium.html')


#### 4. Using hvploting

In [ ]:
%load_ext autoreload
%autoreload 2 


import duckdb 
import geopandas as gpd
import pandas as pd
from config.settings import STORAGE_CONFIG 
from src.h3_system.plot_utils_dev import plot_h3_from_db_fast

import holoviews as hv
hv.extension('bokeh')

In [ ]:
plot_hv_bokeh = plot_h3_from_db_fast(
    h3_cell_ids = h3_cells_to_plot,
    duckdb_path = H3_DUCKDB_PATH,
    popup_fields = ['h3_derived_id', 'state_name','lga_name', 'ward_name', 'confidence_level', 'coverage_percentage'],
    color_by_column=None,
    static_fill_color='#3388ff',
    # line_color="#000000",
    show_markers=False,
    show_legend=False,
    fill_opacity=0.2,   
    width=800,
    height=600
)

plot_hv_bokeh

In [ ]:

# Save the plot to an HTML file
hv.save(plot_hv_bokeh, test_map_path/'hv_plot.html', backend='bokeh') 



### 06. SP AND CUSTOMER DATA

1. Creating Coverage Area About MFCs at Resolution 8

In [ ]:
%load_ext autoreload
%autoreload 2 

import sys
from pathlib import Path 
import geopandas as gpd

from src.get_data import DataFetcher, get_processed_data, get_geojson_data
from src.data.preprocess_data import preprocess_sp_location_mapping

In [ ]:
# 1. Fetch data from db and store locally: 
# - `sp_dim.sql` -> `df_sp_dim.feather`: Fetches stock point dimension data.
# - `sp_location_map.sql` -> `df_sp_location_mapping.feather`: Fetches the mapping of stock points to locations.
# - `get_customer_dim.sql` -> `df_customer_dim.feather`: Fetches customer dimension data.
# - `sp_active_customers.sql` -> `df_sp_active_customers.feather`: Fetches data for active customers associated with stock points.

## Fetch Data from DB
fetcher = DataFetcher(logger=logger, input_dir=str(RAW_DATA_DIR), sql_dir="_sql")
results = fetcher.fetch_all()


# ETA: 5mins

In [ ]:
%load_ext autoreload
%autoreload 2 

from src.data.preprocess_data import preprocess_sp_location_mapping, prepare_sp_and_recent_activated_customers
from src.data.preprocess_data import load_and_preprocess_sp_lga_mapping_data

# 2. Preprocess Sp Location Mapping LGA
preprocess_sp_location_mapping(logger=logger)
prepare_sp_and_recent_activated_customers(logger)

### 07. SP COVERAGE AREA AND CUSTOMER ASSIGNEMENT  


Run complete pipeline  
'''
This will return a dictionary with the following keys: ['territories', 'grid_results', 'assignments', 'optimized_clusters', 'statistics', 'territory_version']
1. territories: A dictionary of stock point territories 
        # Dict[stock_point_id, {  
            'polygon': Union[Polygon, MultiPolygon],  
            'lga_ids': List[str],  
            'is_contiguous': bool,  
            'sub_territories': List[Polygon],  
            'total_area_km2': float,  
            'territory_version': str  
        }]  
  
2. grid_results: A dictionary of grid results for each territory  
        Dict[stock_point_id, {  
                'h3_resolution': int,  
                'h3_cells': Set[str],  
                'clipped_cells': Set[str],   
                'cell_geometries': Dict[str, Polygon],  
                'territory_coverage': float  
        }]  
          
3. assignments: A dictionary of customer assignments to stock points  
        Dict[stock_point_id, assignments_gdf with columns:  
                ['customer_id',   
                'cluster_id', 
                'h3_cell_id', 
                'assignment_confidence', 
                'assignment_tier',
                'geometry']]
        
4. optimized_clusters: A dictionary of optimized clusters for each territory

5. statistics: A dictionary of statistics for each territory

6. territory_version: The version of the territory used in the clustering
'''

In [ ]:
%load_ext autoreload
%autoreload 2 

import pandas as pd
from src.H3SpatialClusterer import H3SpatialClusterer  
from src.get_data import get_processed_data #, get_geojson_data,DataFetcher, 
import pickle
from src.utils import clean_customer_gdf_coordinates
import json
from src.utils import filter_cluster_result_dict
from src.plot_utils import plot_geojson_territory_heatmap

In [ ]:
# 3. Fetch store processed data
# lgas_gdf,  sp_dim_df,  stock_point_lga_map, customers_gdf = get_processed_data()
lgas_gdf, sp_dim_df,  stock_point_lga_map, sp_customers_gdf, recent_customers_gdf  = get_processed_data(logger)

#### Setting Up the Pilot Stock Points

In [ ]:
pilot_2_sps = [1647402,	1647372,	1647108,	1646971,	1647109,	1647033,	
               1646999,	1647391,	1647113,	1647137,	1646991,	1647420,	
               1647141,	1647050,	1647421,	1647436,	1647380             ]
pilot_stock_point_lga_map = stock_point_lga_map[stock_point_lga_map['stock_point_id'].isin(pilot_2_sps)]

#### Set-Up H3SpatialClusterer

In [ ]:
# 1. Initialize with real dataframes
clusterer = H3SpatialClusterer(
    lga_gdf=lgas_gdf,
    sp_dim_df=sp_dim_df[sp_dim_df['stock_point_id'].isin(pilot_2_sps)], 
    stock_point_lga_map=stock_point_lga_map[stock_point_lga_map['stock_point_id'].isin(pilot_2_sps)], 
    customers_gdf=sp_customers_gdf[sp_customers_gdf['stock_point_id'].isin(pilot_2_sps)]
)

#### EDA

In [ ]:
## Add Module src/data/preprocess_data.py
## Preprocess all sp location mapping ----------------------------------------
sp_loc_path = INPUT_BASE_DATA_SOURCES['sp_location_mapping']['local_file_path']
df_sp_location_mapping = pd.read_feather(sp_loc_path)
df_sp_location_mapping.columns = df_sp_location_mapping.columns.str.lower()
df_sp_location_mapping = (df_sp_location_mapping
                            .assign(lga_name_=lambda x: x['lga_name'].str.lower())
                            .query('~lga_name_.str.contains("self|push")', engine='python')
                            .drop(columns=['lga_name_'])
                            .reset_index(drop=True)
                            )

print(df_sp_location_mapping.columns.to_list())
print(len(df_sp_location_mapping))

## Preprocess ng lcda geometric file ----------------------------------------
import geopandas as gpd
lcda_geojson_path = ADMIN_DATA_SOURCES['wards']['standardize_file_path']
lcda_gdf = gpd.read_file(lcda_geojson_path)[[  'state_name', 'state_code','lga_name', 'lga_code','ward_name', 'ward_code']] #shape # (9410, 18)
lcda_gdf.columns = [f'{col}_ng' for col in lcda_gdf.columns]
print(lcda_gdf.columns.to_list())
print(len(lcda_gdf))

# Save to disk
# sp_dim_df.to_excel('./output/base_data_export/sp_dim.xlsx', index=False) 
# df_sp_location_mapping.to_excel('./output/base_data_export/sp_location_mapping.xlsx', index=False) 
# lcda_gdf.to_excel('./output/base_data_export/ng_wards.xlsx', index=False) 

In [ ]:
# help(pd.set_option) 

In [ ]:
# Q1. Table of Stock Point and count of LGAs mapped to them
pd.set_option("display.max_row", None)
pd.set_option("display.max_columns", None)

print(stock_point_lga_map.columns.to_list())
# print("--"*100)
# print(f"Distinct Count of LGA mapped to SPs")
# df_eda_sp_lga_count = (stock_point_lga_map.groupby(['stock_point_id', 'stock_point_name'])
#                         .agg( n_map_lgas = ('lga_id','nunique') )
#                         .sort_values('n_map_lga',ascending=False)
#                         .reset_index() 
#                         )

# print(df_eda_sp_lga_count.n_map_lgas.describe())
# print("--"*100)
# print(df_eda_sp_lga_count[['stock_point_name', 'n_map_lgas']].head(3))

# # 2. How many SP are mapped to same location (lga)
# print("--"*100)
# print(f"# 2. How many SPs were mapped to same location (lga)") 
# df_eda_lga_sp_count = (stock_point_lga_map.groupby(['state_id', 'lga_id', 'state_name', 'lga_name'])
#                         .agg(n_map_sps = ('stock_point_id','nunique') )
#                         .sort_values('n_map_sps',ascending=False)
#                         .reset_index() 
#                         )

# df_eda_sp_lga_map_sp_count = (stock_point_lga_map
#                             .merge(df_eda_lga_sp_count[['state_id', 'lga_id', 'n_map_sps']], on=['state_id', 'lga_id'], how='left')
#                             .sort_values(['n_map_sps','state_id', 'lga_id'], ascending=[False, True, True]))


# print(df_eda_lga_sp_count.n_map_sps.describe())
# print("--"*100)
# print(f'Total Number of SP with same lga mapped to another SP', df_eda_sp_lga_map_sp_count.query('n_map_sps > 1').stock_point_id.nunique(), ' Out of ',df_eda_sp_lga_map_sp_count.stock_point_id.nunique() )
# print(df_eda_lga_sp_count[['state_name', 'lga_name', 'n_map_sps']].head(3))
# print(df_eda_sp_lga_map_sp_count.merge(df_eda_lga_sp_count.iloc[0:1][['state_id','lga_id']])[['stock_point_name','state_name', 'lga_name', 'n_map_sps']])


# --------------------------------------------------------------------------------------------
print("--"*100)
print(f"Evaluating Multiple LGA Mapping to Pilot SPs")
# print(pilot_stock_point_lga_map.columns.to_list())

df_eda_lga_sp_count_pilot = (pilot_stock_point_lga_map
                                    .groupby(['state_id', 'lga_id', 'state_name', 'lga_name'])
                                    .agg(n_map_sps = ('stock_point_id','nunique') )
                                    .sort_values('n_map_sps',ascending=False)
                                    .reset_index() 
                                    )
df_eda_sp_lga_map_sp_count_pilot = (pilot_stock_point_lga_map
                                    .merge(df_eda_lga_sp_count_pilot[['state_id', 'lga_id', 'n_map_sps']], on=['state_id', 'lga_id'], how='left')
                                    .sort_values(['n_map_sps','state_id', 'lga_id'], ascending=[False, True, True])) 
print(df_eda_lga_sp_count_pilot.n_map_sps.describe())
print(df_eda_lga_sp_count_pilot.value_counts('n_map_sps'))
print("--"*100)
print(f'List of SPID with same lga mapped to another SP', df_eda_sp_lga_map_sp_count_pilot.query('n_map_sps > 1').stock_point_id.unique())
print(f'Total Number of SP with same lga mapped to another SP', df_eda_sp_lga_map_sp_count_pilot.query('n_map_sps > 1').stock_point_id.nunique(), ' Out of ',df_eda_sp_lga_map_sp_count_pilot.stock_point_id.nunique() )
print(df_eda_lga_sp_count_pilot[['state_name', 'lga_name','n_map_sps']].drop_duplicates().head(3))
print(df_eda_sp_lga_map_sp_count_pilot.merge(df_eda_lga_sp_count_pilot.iloc[0:2][['state_id','lga_id']])[['stock_point_name','state_name', 'lga_name', 'n_map_sps']])



In [ ]:
df_eda_sp_lga_map_sp_count_pilot

#### Stock Point Coverage Mapping and Clustering

In [ ]:
# Pilot sp list
pilot_sps_lists = list(set(pilot_stock_point_lga_map.stock_point_id) )
stock_point_id = str(pilot_sps_lists[1])
print('Total Pilot SPs', len(pilot_sps_lists))

In [ ]:
print(sp_customers_gdf.shape[0])
print(recent_customers_gdf.shape[0])
print(pilot_stock_point_lga_map .shape[0])

In [ ]:
# 1. Initialize with real dataframes
clusterer = H3SpatialClusterer(
    lga_gdf=lgas_gdf,
    sp_dim_df=sp_dim_df[sp_dim_df['stock_point_id'].isin(pilot_2_sps)], 
    stock_point_lga_map=stock_point_lga_map[stock_point_lga_map['stock_point_id'].isin(pilot_2_sps)], 
    customers_gdf=sp_customers_gdf[sp_customers_gdf['stock_point_id'].isin(pilot_2_sps)]
)

In [ ]:
PILOT_SPS_CLUSTER_R8 =  clusterer.process_all_stock_points(territory_version="v1.2")

# Processing territory for stock point 1647380...
# 📍 Non-contiguous territory detected: 2 sub-territories

In [ ]:
# Phase 1: Territory Definition
territories = clusterer.define_territories()

# # Phase 2: H3 Grid Generation
# grid_results = self.generate_h3_grids(territories)

# # Phase 3: Customer Assignment
# assignments = self.assign_customers_to_clusters(grid_results)

In [ ]:
territories.keys()
territories['1646971']

In [ ]:
PILOT_SP_CLUSTERS_R8_PATH = EXPORTS_DIR /  "PILOT_SPS_CLUSTER_R8.pickle"

# # Save Results as pickle file
# with open(PILOT_SP_CLUSTERS_R8_PATH, 'wb') as f:
#     pickle.dump(PILOT_SPS_CLUSTER_R8, f)
 
# Testing -r8
# Open Results as pickle file
if PILOT_SP_CLUSTERS_R8_PATH.exists():
    with open(PILOT_SP_CLUSTERS_R8_PATH, 'rb') as f:
        PILOT_SPS_CLUSTER_R8 = pickle.load(f)
else:
    raise FileNotFoundError(f"Pickle file not found: {PILOT_SP_CLUSTERS_R8_PATH}")


In [ ]:
PILOT_SPS_CLUSTER_R8.keys()
# ['territories', 'grid_results', 'assignments', 'optimized_clusters', 'statistics', 'territory_version']
# ALL_CLUSTER['territories']

In [ ]:
# # PILOT_SPS_CLUSTER['optimized_clusters']['1646991'] 
# # PILOT_SPS_CLUSTER_FLAT.keys() #['clusters', 'assignments', 'territory_summary', 'territory_cells']

# print(PILOT_SPS_CLUSTER_FLAT['clusters'].stock_point_id.nunique())
# print(PILOT_SPS_CLUSTER_FLAT['territory_cells'].stock_point_id.nunique())

In [ ]:
from src.utils import filter_cluster_result_dict
filtered_result = filter_cluster_result_dict(PILOT_SPS_CLUSTER_R8, pilot_sps_lists) 

filtered_result.keys()

#### Prep sharable Export File

In [ ]:
PILOT_SP_CLUSTERS_R8_PATH = EXPORTS_DIR /  "PILOT_SPS_CLUSTER_R8.pickle"
if PILOT_SP_CLUSTERS_R8_PATH.exists():
    with open(PILOT_SP_CLUSTERS_R8_PATH, 'rb') as f:
        PILOT_SPS_CLUSTER_R8 = pickle.load(f)
else:
    raise FileNotFoundError(f"Pickle file not found: {PILOT_SP_CLUSTERS_R8_PATH}")

In [ ]:
# 4. Export for deployment: ETA: 1 Min
PILOT_SPS_CLUSTER_R8_FLAT = clusterer.export_results(PILOT_SPS_CLUSTER_R8, output_format="csv") 


def extract_coverage_and_assignment_results(clusterer, cluster_result_dict):
    PILOT_SPS_CLUSTER_R8_FLAT = clusterer.export_results(cluster_result_dict, output_format="csv")

    # Stock Point Assignment Summary
    df_output_sp_coverage_cluster = PILOT_SPS_CLUSTER_R8_FLAT['territory_cells']  
    df_output_sp_customer_assignment = PILOT_SPS_CLUSTER_R8_FLAT['assignments']
    df_output_ap_coverage_cluster_summary = PILOT_SPS_CLUSTER_R8_FLAT['clusters']

    df_output_sp_customer_assignment['stock_point_id'] = df_output_sp_customer_assignment['stock_point_id'].astype(int)
    df_output_sp_coverage_cluster['stock_point_id'] = df_output_sp_coverage_cluster['stock_point_id'].astype(int)

    return df_output_sp_coverage_cluster, df_output_sp_customer_assignment, df_output_ap_coverage_cluster_summary

def prepare_sp_assignment_summary(df_output_sp_customer_assignment, sp_dim_df):
    sp_assignment_summary = (df_output_sp_customer_assignment
                            .groupby(['stock_point_id','cluster_id'])['customer_id'].count()
                            .reset_index(name='n_customers')
                            .rename({'cluster_id':'h3_cell'}, axis=1) 
                        ) 
    # Stock Point Coverage - Assignment Summary
    sp_coverage_cluster_and_assignment_summary = (df_output_sp_coverage_cluster
                                                .merge(sp_dim_df[['stock_point_id', 'stock_point_name']] , on='stock_point_id', how='left')
                                                .merge(sp_assignment_summary, how='left', on=['stock_point_id','h3_cell'])
                                                .fillna({'n_customers':0})
                                                .rename({'h3_cell':'cluster_id'}, axis=1)
                                                ) 
    sp_coverage_cluster_and_assignment_summary['n_customers'] = sp_coverage_cluster_and_assignment_summary['n_customers'].astype(int) 

    return sp_assignment_summary


df_output_sp_coverage_cluster, df_output_sp_customer_assignment, df_output_ap_coverage_cluster_summary = extract_coverage_and_assignment_results(clusterer = clusterer, 
                                                                                                                                                 cluster_result_dict = PILOT_SPS_CLUSTER_R8)
sp_coverage_cluster_and_assignment_summary = prepare_sp_assignment_summary(df_output_sp_customer_assignment, sp_dim_df)

 
print(df_output_sp_coverage_cluster.stock_point_id.nunique())
print(sp_coverage_cluster_and_assignment_summary.stock_point_id.nunique())

In [ ]:

## Export to DB
import duckdb 
from config.settings import STORAGE_CONFIG
from src.h3_spatial_system.storage.FastH3DuckDBManager import FastH3DuckDBManager, get_db_summary
     
             
H3_DUCKDB_PATH = STORAGE_CONFIG['h3_duckdb_path']

with FastH3DuckDBManager(resolution=8, db_path=H3_DUCKDB_PATH) as db: 
      # db.upsert_sp_coverage_cells(df_output_sp_coverage_cluster)
      db.upsert_customer_cluster_assignment(df_output_sp_customer_assignment)

In [ ]:
with FastH3DuckDBManager(resolution=8, db_path=H3_DUCKDB_PATH) as db: 
    customer_resolution_summary = db.get_change_summary(days_back=7)
    customer_movement = db.get_customer_movements(days_back=7)
    
customer_movement
customer_resolution_summary    

In [ ]:
# df_output_sp_coverage_cluster.head(2)
# print(df_output_sp_coverage_cluster.columns.to_list()) 
# # ['h3_cell', 'stock_point_id', 'h3_resolution']

# print(df_output_sp_customer_assignment.columns.to_list()) 
# print(df_output_sp_customer_assignment.assignment_tier.value_counts())
# df_output_sp_customer_assignment.head(2)
# ['customer_id', 'cluster_id', 'h3_cell_id', 'assignment_confidence', 'assignment_tier', 'stock_point_id', 'h3_resolution'

# print(df_output_ap_coverage_cluster_summary.columns.to_list())  
# df_output_ap_coverage_cluster_summary.head(2) 
# ['cluster_id', 'h3_resolution', 'h3_cells', 'customer_count', 'parent_cluster_id', 'stock_point_id'] 

In [ ]:
import duckdb
import pandas as pd


_ = get_db_summary()

In [ ]:
conn.close()

In [ ]:
from src.h3_spatial_system.h3_system.FastH3DuckDBManager import FastH3DuckDBManager
H3_DUCKDB_PATH = STORAGE_CONFIG['h3_duckdb_path']

In [ ]:
### Adding Coverage Clustering and Customer Assignment to db

from src.h3_spatial_system.h3_system.FastH3DuckDBManager import FastH3DuckDBManager
H3_DUCKDB_PATH = STORAGE_CONFIG['h3_duckdb_path']
# Added New addrss columns for sp_coverage_cells
with FastH3DuckDBManager(resolution=8, db_path=H3_DUCKDB_PATH) as db: 
      db.upsert_customer_cluster_assignment(df_output_sp_customer_assignment)
      db.upsert_sp_coverage_cells(df_output_sp_coverage_cluster)
#     db._add_sp_coverage_cells_columns()


#     db._add_customer_cluster_assignment_columns()
    

In [ ]:
import duckdb 
H3_DUCKDB_PATH = STORAGE_CONFIG['h3_duckdb_path']
conn = duckdb.connect(H3_DUCKDB_PATH)

# Install and load the httpfs extension
# conn.execute("INSTALL 'httpfs';")
conn.execute("LOAD 'httpfs';")

# Load the parquet extension if needed
conn.execute("LOAD 'parquet';")


In [ ]:
## Customer Assignment Table Enhanced
df_output_sp_customer_assignment_enhanced = conn.execute("""SELECT 
                a.stock_point_id, c.stock_point_name, a.customer_id, cluster_id, h3_derived_id cluster_code, 
                ROUND(assignment_confidence * 100, 2) assignment_confidence,
                CASE WHEN assignment_tier = 'h3_inclusion' THEN 'within cluster' 
                    WHEN assignment_tier = 'manual_review' THEN 'manual review'
                ELSE assignment_tier END AS assignment_tier,
                d.business_id,  d.contact_name, d.customer_status, kyc_capture_status, agent_id, agent_name, 
                d.state_name as customer_state_name,	
                d.town_name as customer_town_name,	
                d.city_name as customer_city_name
                FROM df_output_sp_customer_assignment a
                LEFT JOIN h3_cells b ON b.h3_index = a.cluster_id
                LEFT JOIN sp_dim_df c ON c.stock_point_id = a.stock_point_id 
                LEFT JOIN read_parquet('/home/bt/project/demand_engine/StockPoint_Clustering_and_Routing/data/processed/df_processed_customer_dim.parquet') d ON d.customer_id = a.customer_id             
                -- LIMIT 2
                """).df()

print(PILOT_SPS_CLUSTER_FLAT['assignments'].stock_point_id.nunique())
print(df_output_sp_customer_assignment_enhanced.stock_point_id.nunique())
df_output_sp_customer_assignment_enhanced.head(2)

In [ ]:
print(len(df_output_sp_customer_assignment))
print(len(df_output_sp_customer_assignment_enhanced))

In [ ]:
# print(conn.execute('SELECT * FROM h3_cells LIMIT 1').df().columns.to_list())
# ['h3_index', 'resolution', 'centroid_lat', 'centroid_lng', 'polygon_wkt', 'boundary_json', 
#  'latlng_json', 'polygon_area', 'num_vertices', 'error', 'created_at', 'h3_derived_id', 
#  'grid_position_id', 'primary_address_id', 'country_code', 'country_name', 'state_code', 
#  'state_name', 'lga_code', 'lga_name', 'ward_code', 'ward_name', 'confidence_level', 
#  'coverage_percentage', 'area_km2']
from src.utils import calculate_distance_km
sp_coverage_cluster_and_assignment_summary_enhanced = conn.execute('''SELECT 
                    a.stock_point_id, a.stock_point_name,
                    a.cluster_id, h3_derived_id as cluster_code, 
                    a.n_customers as customer_count,
                    confidence_level as cluster_address_level, 
                    state_name as cluster_state_name, lga_name as cluster_lga_name, ward_name as cluster_ward_name,
                    --- Add coord columns if needed,
                    centroid_lat as cluster_centroid_lat, 
                    centroid_lng as cluster_centroid_lng,
                    latitude as sp_lat, 
                    longitude as sp_lng
                FROM sp_coverage_cluster_and_assignment_summary a
                LEFT JOIN  sp_dim_df b ON b.stock_point_id = a.stock_point_id
                LEFT JOIN h3_cells b ON b.h3_index = a.cluster_id 
            ''').df()

sp_coverage_cluster_and_assignment_summary_enhanced['cluster_sp_dist_km'] = (sp_coverage_cluster_and_assignment_summary_enhanced
                                                                             .apply(lambda row: calculate_distance_km(row['cluster_centroid_lat'], row['cluster_centroid_lng'], 
                                                                                                                      row['sp_lat'], row['sp_lng']), axis=1)
                                                                            )
get_cluster_status = lambda dist: 'Undefined' if dist is None else 'Within 7km' if dist <= 7 else 'Above 7km'
sp_coverage_cluster_and_assignment_summary_enhanced['cluster_sp_dist_status'] = (sp_coverage_cluster_and_assignment_summary_enhanced['cluster_sp_dist_km']
                                                                             .apply(lambda x: get_cluster_status(x))
                                                                            )    
try:
    drp_cols = [ 'cluster_centroid_lat','cluster_centroid_lng', 'sp_lat', 'sp_lng']
    sp_coverage_cluster_and_assignment_summary_enhanced.drop(drp_cols, axis=1, inplace=True)
except Exception as e:
    print(e) 


print(PILOT_SPS_CLUSTER_FLAT['territory_cells'].stock_point_id.nunique())
print(sp_coverage_cluster_and_assignment_summary.stock_point_id.nunique())
print(sp_coverage_cluster_and_assignment_summary_enhanced.stock_point_id.nunique())

sp_coverage_cluster_and_assignment_summary_enhanced.sample(2) 

In [ ]:
# Create a Pandas Excel writer using openpyxl as the engine
with pd.ExcelWriter(OUTPUT_DIR / 'pilot 2/coverage_cluster_and_customer_assignment.xlsx', engine='openpyxl') as writer: 
    sp_coverage_cluster_and_assignment_summary_enhanced.to_excel(writer, sheet_name='coverage_cluster', index=False)
    df_output_sp_customer_assignment_enhanced.to_excel(writer, sheet_name='assignment', index=False)

#### Export Customer Assignment to DuckDB

In [ ]:
conn = duckdb.connect(H3_DUCKDB_PATH)

conn.execute("SHOW TABLES").fetchdf() 
conn.execute("DROP TABLE IF EXISTS customer_cluster_assignment").fetchdf() 
# conn.execute("DESCRIBE customer_cluster_assignment;").fetchdf() 

print(conn.execute("SELECT COUNT(*) FROM  h3_cells").fetchone()[0] )
print(conn.execute("SELECT COUNT(*) FROM  sp_coverage_cells;").fetchone()[0])
print(conn.execute("SELECT COUNT(*) FROM  customer_cluster_assignment;").fetchone()[0])


conn.close()

# Data Migration

In [ ]:
# pip install dlt[mssql] duckdb

In [ ]:
import duckdb
from config.settings import STORAGE_CONFIG         
H3_DUCKDB_PATH = STORAGE_CONFIG['h3_duckdb_path'] 

with duckdb.connect(H3_DUCKDB_PATH) as conn:
    print(conn.execute("SHOW TABLES").fetchdf() )
    print(conn.execute("DESCRIBE stockpoint_h3_coverage; ").df())
    print(conn.execute("SELECT COUNT(*) FROM stockpoint_h3_coverage; ").df())
    df_assignment_schema = conn.execute("DESCRIBE customer_stockpoint_cluster_assignment; ").df()
    print(conn.execute("SELECT COUNT(*) FROM customer_stockpoint_cluster_assignment; ").df())
    df_h3_cells_schema = conn.execute("DESCRIBE h3_cells; ").df()
    df_h3_cells = conn.execute("""SELECT 
                                h3_index as h3_cell, resolution, centroid_lat, centroid_lng, 
                                created_at, h3_derived_id, 
                                country_code, country_name, state_code, state_name, lga_code,
                                lga_name, ward_code, ward_name, confidence_level,
                                coverage_percentage, area_km2 
                                FROM h3_cells
                                WHERE (confidence_level IS NOT NULL AND confidence_level <> 'manual_review')
                                        AND resolution=8; 
                            """).df()
    print(conn.execute("SELECT COUNT(*) FROM h3_cells; ").df())
    print(conn.execute("SELECT DISTINCT confidence_level FROM h3_cells; ").df())
    print(conn.execute("""SELECT COUNT(*) 
                       FROM h3_cells 
                       WHERE (confidence_level IS NOT NULL AND confidence_level <> 'manual_review')
                       AND resolution=8; 
                       """).df())
    

                                     name
0  customer_stockpoint_cluster_assignment
1                                h3_cells
2                  stockpoint_h3_coverage
3              stockpoint_h3_coverage_log


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   count_star()
0       2467165
  confidence_level
0    manual_review
1    boundary_case
2             None
3        confident
   count_star()
0       1354919


,column_name,column_type,null,key,default
0,h3_index,VARCHAR,NO,PRI,None
1,resolution,TINYINT,NO,None,None
2,centroid_lat,DOUBLE,YES,None,None
3,centroid_lng,DOUBLE,YES,None,None
4,polygon_wkt,VARCHAR,YES,None,None
5,boundary_json,VARCHAR,YES,None,None
6,latlng_json,VARCHAR,YES,None,None
7,polygon_area,DOUBLE,YES,None,None
8,num_vertices,SMALLINT,YES,None,None
9,error,VARCHAR,YES,None,None


In [14]:
# help(pd.set_option)

In [24]:
import pandas as pd
pd.set_option('display.max_columns', None)
df_h3_cells.value_counts(['confidence_level'])
df_h3_cells.head(2)

,h3_index,resolution,centroid_lat,centroid_lng,created_at,h3_derived_id,country_code,country_name,state_code,state_name,lga_code,lga_name,ward_code,ward_name,confidence_level,coverage_percentage,area_km2
0,8858c6cabdfffff,8,7.875519,9.232290,2025-08-07 03:10:08.777,NG-TA-35014-TRSWKR01-Z9X6133,NG,Nigeria,TA,Taraba,35014,Wukari,TRSWKR01,Akwana,confident,100.0,0.676382
1,8858f5474dfffff,8,9.402631,8.220323,2025-08-07 03:10:08.777,NG-KD-19007-KD0706-QTNL6NZ,NG,Nigeria,KD,Kaduna,19007,Jema'A,KD0706,Jagindi,confident,100.0,0.688050


In [ ]:
h3_cells_selcols = ['h3_index', 'resolution', 'centroid_lat', 'centroid_lng', 
                    'created_at', 'h3_derived_id', 
                    'country_code', 'country_name', 'state_code', 'state_name', 'lga_code',
                    'lga_name', 'ward_code', 'ward_name', 'confidence_level',
                    'coverage_percentage', 'area_km2']

In [ ]:
%load_ext autoreload
%autoreload 2 
from src.data_migration.migrate_duckdb_to_sqlsever import migrate_stockpoint_h3_coverage

In [ ]:
from src.data_migration.migrate_duckdb_to_sqlsever import migrate_stockpoint_h3_coverage
info = migrate_stockpoint_h3_coverage()

In [30]:
%load_ext autoreload
%autoreload 2 


from src.data_migration.migrate_duckdb_to_sqlsever import (
    migrate_stockpoint_h3_coverage,
    migrate_customer_assignment, 
    migrate_h3_cells,
    migrate_all_tables,
    validate_duckdb_setup
)

# Each can be called independently
# migrate_stockpoint_h3_coverage()  # Works standalone
# migrate_customer_assignment()     # Works standalone  
# info = migrate_all_tables()             # Works standalone

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [31]:
# Migrate h3_cells

info_h3_migration = migrate_h3_cells()

Starting migration for: _h3


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Migration completed successfully!
Pipeline run ID: N/A


In [29]:
info_h3_migration

LoadInfo(pipeline=<dlt.pipeline(pipeline_name='duckdb_to_sqlserver', destination='mssql', dataset_name='gis_analysis', default_schema_name='duckdb_source', schema_names=['combined_source', 'duckdb_source', 'h3_cells_source'], pipelines_dir='/home/bt/.dlt/pipelines', working_dir='/home/bt/.dlt/pipelines/duckdb_to_sqlserver')>, metrics={'1755245723.8367755': [{'started_at': DateTime(2025, 8, 15, 8, 15, 24, 69827, tzinfo=Timezone('UTC')), 'finished_at': DateTime(2025, 8, 15, 8, 15, 51, 45347, tzinfo=Timezone('UTC')), 'job_metrics': {'h3_cells.0c2d4c9fb0.insert_values.gz': LoadJobMetrics(job_id='h3_cells.0c2d4c9fb0.insert_values.gz', file_path='/home/bt/.dlt/pipelines/duckdb_to_sqlserver/load/normalized/1755245723.8367755/started_jobs/h3_cells.0c2d4c9fb0.0.insert_values.gz', table_name='h3_cells', started_at=DateTime(2025, 8, 15, 8, 15, 31, 383301, tzinfo=Timezone('UTC')), finished_at=DateTime(2025, 8, 15, 8, 15, 39, 153402, tzinfo=Timezone('UTC')), state='completed', remote_url=None), 'h3